<a href="https://colab.research.google.com/github/rafaellopesdesa/nsbi-lhc-toolkit/blob/ml4hep_school_tutorial/workshops/ml4hep_tifr_colab/Exercise_9c_SBIBM_hybrid.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Exercise 9c — aggregate hybrid comparison

Run this after any subset of the ten task notebooks. It validates each task's profile, seed, and run-tag identity; partial campaigns are allowed and visibly marked. The figures compare the modest flow, one multiclass correction, and two separate binary corrections across posterior, predictive-data, and predictive-joint metrics. A dedicated delta heatmap answers the main question directly: where does multiclass factorization improve or degrade C2ST relative to separate binary estimators?

The collector also relates accuracy to posterior and predictive importance-weight efficiency, because a visually improved distribution with collapsing ESS is not a robust hybrid result.


In [ ]:
import os, sys, subprocess
from pathlib import Path

PROFILE = "TUTORIAL"  # must match the task notebooks
SEEDS = [31082026]
USE_DRIVE = True

if "google.colab" in sys.modules:
    if USE_DRIVE:
        from google.colab import drive
        if not Path("/content/drive/MyDrive").exists():
            drive.mount("/content/drive")
        ARTIFACT_ROOT = Path("/content/drive/MyDrive/hybrid_nsbi_ml/exercise_9c_SBIBM_hybrid")
    else:
        ARTIFACT_ROOT = Path("/content/exercise_9c_SBIBM_hybrid_artifacts")
    repository = Path("/content/nsbi-lhc-toolkit")
    if not (repository / ".git").is_dir():
        clone_env = os.environ.copy()
        clone_env["GIT_LFS_SKIP_SMUDGE"] = "1"
        subprocess.run([
            "git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
            "--branch", "ml4hep_school_tutorial",
            "https://github.com/rafaellopesdesa/nsbi-lhc-toolkit.git", str(repository),
        ], check=True, env=clone_env)
    else:
        subprocess.run(["git", "-C", str(repository), "pull", "--ff-only", "origin", "ml4hep_school_tutorial"], check=True)
    subprocess.run(["git", "-C", str(repository), "sparse-checkout", "set", "src", "workshops/ml4hep_tifr_colab"], check=True)
    tutorial = repository / "workshops" / "ml4hep_tifr_colab"
    sys.path.insert(0, str(tutorial))
    os.chdir(tutorial)
else:
    ARTIFACT_ROOT = Path.cwd() / "exercise_9c_SBIBM_hybrid_artifacts"
    for candidate in [Path.cwd(), Path.cwd() / "workshops" / "ml4hep_tifr_colab"]:
        if (candidate / "utils_exercise9c_aggregate.py").exists():
            sys.path.insert(0, str(candidate.resolve()))
            break

print("Reading artifacts from:", ARTIFACT_ROOT)


In [ ]:
from IPython.display import display
from utils_exercise9c_aggregate import render_aggregate

COMBINED_METRICS, TASK_STATUS = render_aggregate(
    ARTIFACT_ROOT, profile=PROFILE, seeds=SEEDS
)
display(TASK_STATUS.style.hide(axis="index"))
if not COMBINED_METRICS.empty:
    display(COMBINED_METRICS.style.format(precision=4).hide(axis="index"))
